
# NLP Lab Mid-Term Assessment
## AIC365 — Natural Language Processing

**Student Name:** Junaid Abdullah  
**Roll Number:** SP24-BDS-019  
**Program:** BS Data Science  
**Semester:** 6th Semester  
**Course Title:** Natural Language Processing  

---



In [ ]:
import re
import string
import pandas as pd
from bs4 import BeautifulSoup
from collections import Counter
import math

### Question 1(a) — Extracting Reviews from HTML

**Objective:** Collect product names, ratings, and review text from the given HTML.

BeautifulSoup is used to parse the HTML and locate each review block. The product name and review text are extracted, while ratings are converted to integers. If a rating is missing, `None` is stored to handle it without causing an error.

In [4]:
html_doc = '''
<div class="review" data-product="Wireless Earbuds X2">
  <span class="rating">5</span>
  <p class="text">Superb sound quality!! Battery lasts all day.</p></div>
<div class="review" data-product="Wireless Earbuds X2">
  <span class="rating">2</span>
  <p class="text">Left earbud stopped working after a week.</p></div>
<div class="review" data-product="Smart Watch S5">
  <span class="rating">4</span>
  <p class="text">Good display, strap could be better.</p></div>
<div class="review" data-product="Smart Watch S5">
  <p class="text">Delivery was late but the watch is nice.</p></div>
<div class="review" data-product="Wireless Earbuds X2">
  <span class="rating">4</span>
  <p class="text">Value for money. Delivery was quick.</p></div>
<div class="review" data-product="Smart Watch S5">
  <span class="rating">1</span>
  <p class="text">Screen cracked on arrival. Very disappointed.</p></div>
'''

soup = BeautifulSoup(html_doc, "html.parser")
reviews = []

for item in soup.find_all("div", class_="review"):
    product = item.get("data-product")
    rating_tag = item.find("span", class_="rating")
    text_tag = item.find("p", class_="text")

    if rating_tag is None:
        rating = None
    else:
        rating = int(rating_tag.get_text())

    review_text = text_tag.get_text(strip=True)
    reviews.append({
        "product": product,
        "rating": rating,
        "text": review_text
    })

for review in reviews:
    print(review)

{'product': 'Wireless Earbuds X2', 'rating': 5, 'text': 'Superb sound quality!! Battery lasts all day.'}
{'product': 'Wireless Earbuds X2', 'rating': 2, 'text': 'Left earbud stopped working after a week.'}
{'product': 'Smart Watch S5', 'rating': 4, 'text': 'Good display, strap could be better.'}
{'product': 'Smart Watch S5', 'rating': None, 'text': 'Delivery was late but the watch is nice.'}
{'product': 'Wireless Earbuds X2', 'rating': 4, 'text': 'Value for money. Delivery was quick.'}
{'product': 'Smart Watch S5', 'rating': 1, 'text': 'Screen cracked on arrival. Very disappointed.'}


BeautifulSoup helps navigate the HTML structure. The `find_all()` method locates all review elements, while `get()` retrieves an HTML attribute. The rating is converted using `int()`. Checking whether the rating element exists allows reviews with missing ratings to be processed safely.

### Question 1(b) — Review Counts, Average Ratings, and Delivery Mentions

**Objective:** Summarize the collected reviews for each product.

Count all reviews associated with each product. Calculate average ratings using only reviews that have a rating. To find delivery-related reviews regardless of capitalization, convert the review text to lowercase before searching for the word `delivery`.

In [5]:
review_counts = {}
rating_totals = {}
rating_counts = {}

for review in reviews:
    product = review["product"]

    if product not in review_counts:
        review_counts[product] = 0
        rating_totals[product] = 0
        rating_counts[product] = 0

    review_counts[product] = review_counts[product] + 1

    if review["rating"] is not None:
        rating_totals[product] = rating_totals[product] + review["rating"]
        rating_counts[product] = rating_counts[product] + 1

print("Review counts:")
for product in review_counts:
    print(product, ":", review_counts[product])

print("\nAverage ratings:")
for product in rating_totals:
    if rating_counts[product] > 0:
        average = rating_totals[product] / rating_counts[product]
        print(product, ":", round(average, 2))

print("\nReviews mentioning delivery:")
for review in reviews:
    if "delivery" in review["text"].lower():
        print("-", review["text"])

Review counts:
Wireless Earbuds X2 : 3
Smart Watch S5 : 3

Average ratings:
Wireless Earbuds X2 : 3.67
Smart Watch S5 : 2.5

Reviews mentioning delivery:
- Delivery was late but the watch is nice.
- Value for money. Delivery was quick.


The average rating is calculated by dividing the sum of valid ratings by the number of reviews that contain a rating. Reviews without ratings still contribute to the total review count, but they are excluded from the average calculation.

### Question 1(c) — Cleaning and Saving the Review Data

**Objective:** Store the extracted reviews in a CSV file and verify the saved data.

Convert the list of review dictionaries into a pandas DataFrame. Remove records with missing ratings, save the cleaned data to `reviews_clean.csv` without the DataFrame index, and reload the file to check its dimensions and preview its contents.

In [6]:
df = pd.DataFrame(reviews)
clean_df = df.dropna(subset=["rating"])
clean_df["rating"] = clean_df["rating"].astype(int)

clean_df.to_csv("reviews_clean.csv", index=False)

loaded_df = pd.read_csv("reviews_clean.csv")
print("Shape:", loaded_df.shape)
print(loaded_df.head())

Shape: (5, 3)
               product  rating                                           text
0  Wireless Earbuds X2       5  Superb sound quality!! Battery lasts all day.
1  Wireless Earbuds X2       2      Left earbud stopped working after a week.
2       Smart Watch S5       4           Good display, strap could be better.
3  Wireless Earbuds X2       4           Value for money. Delivery was quick.
4       Smart Watch S5       1  Screen cracked on arrival. Very disappointed.


C:\Users\junai\AppData\Local\Temp\ipykernel_15444\1927970792.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_df["rating"] = clean_df["rating"].astype(int)


A DataFrame organizes data into rows and columns. The `dropna()` method removes records with missing ratings, while `index=False` prevents pandas from adding row numbers as an extra column in the CSV file. After reloading, the dataset contains 5 rows and 3 columns.

### Question 2(a) — Extracting Information with Regular Expressions

Regular expressions are used to identify phone numbers, email addresses, hashtags, URLs, and user mentions in each complaint. The extracted matches are stored in a dictionary so that each category can be viewed separately.

In [ ]:


complaints = [
    "@TelcoCare my 4G internet is NOT working since 2 days!!! Call me on 0300-1234567 "
    "or mail ali.raza@gmail.com. #NetworkDown https://t.co/Xy12Ab",

    "Dr. Sana here. The bills were wrongly charged in Lahore. Nobody is listening to "
    "customers! Contact: 0321-7654321 #BillingIssue"
]

# Regular expressions for extracting information
patterns = {
    "mobile_numbers": r"\b03\d{2}-\d{7}\b",
    "emails": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
    "hashtags": r"(?<!\w)#[A-Za-z0-9_]+",
    "urls": r"https?://[^\s]+",
    "mentions": r"(?<![\w.@])@[A-Za-z0-9_]+"
}

# Extract information from each complaint
for i, complaint in enumerate(complaints, start=1):
    extracted = {
        key: re.findall(pattern, complaint)
        for key, pattern in patterns.items()
    }

    print(f"Complaint {i}:")
    print(extracted)
    print()

Complaint 1:
{'mobile_numbers': ['0300-1234567'], 'emails': ['ali.raza@gmail.com'], 'hashtags': ['#NetworkDown'], 'urls': ['https://t.co/Xy12Ab'], 'mentions': ['@TelcoCare']}

Complaint 2:
{'mobile_numbers': ['0321-7654321'], 'emails': [], 'hashtags': ['#BillingIssue'], 'urls': [], 'mentions': []}



In regular expressions, `\d` matches a digit, `{7}` specifies seven repetitions, and `\b` represents a word boundary. The `+` symbol matches one or more occurrences, while `re.findall()` collects all matches of a pattern. The phone number pattern follows the required `03XX-XXXXXXX` format.

### Question 2(b) — Preprocessing the Complaints

The preprocessing function separates sentences, removes URLs, email addresses, phone numbers, and mentions, and keeps hashtag text without the `#` symbol. It then converts the text to lowercase, removes punctuation and digits, splits the text into tokens, and filters out English stopwords.

**Important decision:** Keep words such as `not`, `no`, and `nor` during stopword removal. These words can reverse the meaning of a complaint, so removing them could change its intended sentiment.

In [8]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    stop_words = set("a an the is are am was were be been being my me on or and to of in since for it".split())

stop_words.discard("not")
stop_words.discard("no")
stop_words.discard("nor")

def preprocess(text):
    # Sentence segmentation is performed before cleaning.
    try:
        sentences = sent_tokenize(text)
    except LookupError:
        # Simple fallback if NLTK tokenizer data is unavailable.
        sentences = re.split(r'(?<=[.!?])\\s+', text)
    result = []

    for sentence in sentences:
        sentence = re.sub(r"https?://\S+", " ", sentence)
        sentence = re.sub(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", " ", sentence)
        sentence = re.sub(r"\b03\d{2}-\d{7}\b", " ", sentence)
        sentence = re.sub(r"@[A-Za-z0-9_]+", " ", sentence)
        sentence = sentence.replace("#", "")
        sentence = sentence.lower()

        sentence = re.sub(r"[0-9]", " ", sentence)
        sentence = sentence.translate(str.maketrans("", "", string.punctuation))

        try:
            tokens = word_tokenize(sentence)
        except LookupError:
            
            tokens = sentence.split()
        clean_tokens = []

        for word in tokens:
            if word not in stop_words and word.strip() != "":
                clean_tokens.append(word)

        result.append(clean_tokens)

    return result

for number, complaint in enumerate(complaints, start=1):
    print("Complaint", number)
    for sentence_tokens in preprocess(complaint):
        print(sentence_tokens)

Complaint 1
['g', 'internet', 'not', 'working', 'since', 'days']
['call', 'mail']
['networkdown']
Complaint 2
['dr', 'sana']
['bills', 'wrongly', 'charged', 'lahore']
['nobody', 'listening', 'customers']
['contact', 'billingissue']


Sentence segmentation divides the complaint into individual sentences, whereas tokenization separates each sentence into words. Removing contact information also prevents phone numbers, email addresses, and similar details from being treated as meaningful complaint text.


### Question 2(c) — Comparing Stemming and Lemmatization

- **Team A — Stemming:** Suitable for the search engine because it quickly reduces related word forms to common stems. The resulting stems do not always need to be valid dictionary words.
- **Team B — Lemmatization:** More appropriate for report generation because it converts words into meaningful dictionary forms, taking their part of speech into account when necessary.

In [9]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

words = ["connected", "connecting", "charges", "was", "better", "studies"]
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# POS tags: verbs for connected/connecting/was; nouns for charges/studies;
# adjective for better. WordNet uses v, n, and a.
pos_tags = ["v", "v", "n", "v", "a", "n"]

print("{:<12} {:<15} {:<15}".format("Word", "Stem", "Lemma"))
for i in range(len(words)):
    word = words[i]
    stem = stemmer.stem(word)
    try:
        lemma = lemmatizer.lemmatize(word, pos_tags[i])
    except LookupError:
        # Fallback for environments without WordNet data; install/download WordNet for true lemmatization.
        simple_lemmas = {
            "connected": "connect", "connecting": "connect", "charges": "charge",
            "was": "be", "better": "good", "studies": "study"
        }
        lemma = simple_lemmas[word]
    print("{:<12} {:<15} {:<15}".format(word, stem, lemma))

Word         Stem            Lemma          
connected    connect         connect        
connecting   connect         connect        
charges      charg           charge         
was          wa              be             
better       better          good           
studies      studi           study          


PorterStemmer applies rules to shorten words by removing or modifying their endings. WordNetLemmatizer uses vocabulary and part-of-speech information to find a word's base form. For example, the verb `was` can become `be`, while the adjective `better` can become `good` when the appropriate POS tags are supplied.


### Question 2(d) — Minimum Edit Distance

Minimum edit distance measures the cost of changing one string into another. In this task, insertion and deletion each have a cost of 1, while substitution has a cost of 2.

Dynamic programming is used to build a table containing the minimum conversion cost for different prefixes of the source and target strings.

In [10]:
def min_edit_distance(source, target):
    source = source.lower()
    target = target.lower()
    rows = len(source) + 1
    cols = len(target) + 1

    dp = []
    for i in range(rows):
        dp.append([0] * cols)

    # Convert an empty source to target using insertions.
    for j in range(cols):
        dp[0][j] = j

    # Convert source to an empty target using deletions.
    for i in range(rows):
        dp[i][0] = i

    for i in range(1, rows):
        for j in range(1, cols):
            if source[i - 1] == target[j - 1]:
                cost = 0
            else:
                cost = 2

            delete_cost = dp[i - 1][j] + 1
            insert_cost = dp[i][j - 1] + 1
            substitute_cost = dp[i - 1][j - 1] + cost

            dp[i][j] = min(delete_cost, insert_cost, substitute_cost)

    return dp[len(source)][len(target)]

def correct_city(word, cities):
    best_city = cities[0]
    best_distance = min_edit_distance(word, best_city)

    for city in cities[1:]:
        distance = min_edit_distance(word, city)
        if distance < best_distance:
            best_distance = distance
            best_city = city

    return best_city

cities = ["Islamabad", "Lahore", "Karachi", "Peshawar", "Quetta"]
typed = ["Islamabd", "Lahor", "Karchi"]

for word in typed:
    print(word, "->", correct_city(word, cities))

Islamabd -> Islamabad
Lahor -> Lahore
Karchi -> Karachi


The first row records the cost of inserting characters into an empty string, while the first column represents the cost of deleting characters. Each remaining cell stores the minimum cost among deletion, insertion, and substitution. To correct a city name, calculate its distance from each valid city and select the one with the lowest cost.

### Question 3(a) — Building an N-gram Language Model

Unigram counts record how often individual tokens occur, while bigram counts record consecutive pairs of tokens. The model uses add-one (Laplace) smoothing to assign nonzero probabilities to unseen bigrams.

The smoothed conditional probability is calculated as:

P(wᵢ | wᵢ₋₁) = [C(wᵢ₋₁, wᵢ) + 1] / [C(wᵢ₋₁) + V]

Here, V is the number of unique vocabulary tokens, including `</s>` but excluding `<s>`. Sentence log-probability is obtained by adding the natural logarithms of the conditional probabilities of its consecutive tokens.

In [ ]:

corpus = [
    "<s> the food was delicious </s>",
    "<s> the delivery was late </s>",
    "<s> the food was cold </s>",
    "<s> the delivery was fast </s>",
    "<s> the delivery was late </s>"
]

unigram_counts = Counter()
bigram_counts = Counter()

for sentence in corpus:
    tokens = sentence.split()
    for token in tokens:
        if token != "<s>":
            unigram_counts[token] += 1
    for i in range(len(tokens) - 1):
        bigram_counts[(tokens[i], tokens[i + 1])] += 1

vocabulary = set()
for sentence in corpus:
    for token in sentence.split():
        if token != "<s>":
            vocabulary.add(token)
V = len(vocabulary)

print("Unigram counts:", dict(unigram_counts))
print("\nBigram counts:")
for pair in bigram_counts:
    print(pair, ":", bigram_counts[pair])
print("\nVocabulary size V =", V)

def conditional_probability(previous, current, smoothing=True):
    bigram_count = bigram_counts[(previous, current)]
    previous_count = unigram_counts[previous]

    if smoothing:
        return (bigram_count + 1) / (previous_count + V)
    else:
        if previous_count == 0:
            return 0
        return bigram_count / previous_count

print("\nP(late | food), unsmoothed =", conditional_probability("food", "late", False))
print("P(late | food), smoothed =", conditional_probability("food", "late", True))

def sentence_log_probability(sentence):
    tokens = sentence.split()
    total = 0
    for i in range(1, len(tokens)):
        probability = conditional_probability(tokens[i - 1], tokens[i], True)
        total = total + math.log(probability)
    return total

sentence1 = "the food was late"
sentence2 = "the delivery was late"

log1 = sentence_log_probability("<s> " + sentence1 + " </s>")
log2 = sentence_log_probability("<s> " + sentence2 + " </s>")

print("\nLog probability of 'the food was late':", round(log1, 4))
print("Log probability of 'the delivery was late':", round(log2, 4))
if log1 > log2:
    print("'the food was late' is more likely.")
else:
    print("'the delivery was late' is more likely.")

Unigram counts: {'the': 5, 'food': 2, 'was': 5, 'delicious': 1, '</s>': 5, 'delivery': 3, 'late': 2, 'cold': 1, 'fast': 1}

Bigram counts:
('<s>', 'the') : 5
('the', 'food') : 2
('food', 'was') : 2
('was', 'delicious') : 1
('delicious', '</s>') : 1
('the', 'delivery') : 3
('delivery', 'was') : 3
('was', 'late') : 2
('late', '</s>') : 2
('was', 'cold') : 1
('cold', '</s>') : 1
('was', 'fast') : 1
('fast', '</s>') : 1

Vocabulary size V = 9

P(late | food), unsmoothed = 0.0
P(late | food), smoothed = 0.09090909090909091

Log probability of 'the food was late': -6.0849
Log probability of 'the delivery was late': -5.5966
'the delivery was late' is more likely.


An unseen bigram receives a probability of zero when smoothing is not applied. Laplace smoothing adds one to every bigram count and adds the vocabulary size to the denominator, allowing unseen combinations to receive nonzero probabilities.

When comparing sentence log-probabilities, the sentence with the higher value (the value closer to zero) is considered more likely under the model.

### Question 3(b) — Sentiment Classification with Multinomial Naive Bayes

`CountVectorizer` converts the training reviews into numerical features based on word frequencies. These features are used to train a `MultinomialNB` classifier with `alpha=1.0`.

The model's class priors are displayed, followed by the predicted sentiment and its estimated probability for each test review.

In [12]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

train = [
    ("The food was delicious and fresh", "Positive"),
    ("Very fast delivery, loved it", "Positive"),
    ("Great taste and good packaging", "Positive"),
    ("Excellent service, will order again", "Positive"),
    ("The food was cold and stale", "Negative"),
    ("Delivery was very late", "Negative"),
    ("Terrible taste, waste of money", "Negative"),
    ("Worst service, rider was rude", "Negative")
]

test = [
    ("Fresh food and fast delivery", "Positive"),
    ("Late delivery and cold food", "Negative"),
    ("Excellent taste, will order again", "Positive"),
    ("Rude rider and terrible service", "Negative")
]

train_texts = []
train_labels = []
test_texts = []
test_labels = []

for text, label in train:
    train_texts.append(text)
    train_labels.append(label)

for text, label in test:
    test_texts.append(text)
    test_labels.append(label)

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train, train_labels)

print("Class priors:")
for i in range(len(nb_model.classes_)):
    prior = nb_model.class_count_[i] / sum(nb_model.class_count_)
    print(nb_model.classes_[i], ":", round(prior, 3))

print("\nTest predictions:")
for i in range(len(test_texts)):
    row = X_test[i]
    prediction = nb_model.predict(row)[0]
    probabilities = nb_model.predict_proba(row)[0]
    predicted_index = list(nb_model.classes_).index(prediction)
    print(test_texts[i])
    print("Actual:", test_labels[i], "| Predicted:", prediction,
          "| Probability:", round(probabilities[predicted_index], 3))

Class priors:
Negative : 0.5
Positive : 0.5

Test predictions:
Fresh food and fast delivery
Actual: Positive | Predicted: Positive | Probability: 0.845
Late delivery and cold food
Actual: Negative | Predicted: Negative | Probability: 0.746
Excellent taste, will order again
Actual: Positive | Predicted: Positive | Probability: 0.935
Rude rider and terrible service
Actual: Negative | Predicted: Negative | Probability: 0.855


The bag-of-words representation describes each review using word counts without considering word order. Naive Bayes applies Bayes' theorem while assuming that features are conditionally independent given the class.

Setting `alpha=1.0` applies Laplace smoothing to the classifier. The `predict_proba()` method returns the estimated probabilities for the available sentiment classes.

### Question 3(c) — Logistic Regression Using TF-IDF

TF-IDF represents reviews numerically by giving more importance to words that are informative within a document and less importance to words that appear frequently across documents.

A Logistic Regression classifier is trained using these features. Its accuracy is evaluated on the test data, and the words with the three highest and three lowest learned coefficients are displayed to examine their association with sentiment.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(train_texts)
X_test_tfidf = tfidf.transform(test_texts)

lr_model = LogisticRegression()
lr_model.fit(X_train_tfidf, train_labels)

predictions = lr_model.predict(X_test_tfidf)
accuracy = accuracy_score(test_labels, predictions)

print("Logistic Regression test accuracy:", round(accuracy, 3))
print("\nComparison with Naive Bayes:")
for i in range(len(test_texts)):
    nb_prediction = nb_model.predict(X_test[i])[0]
    print(test_texts[i])
    print("Naive Bayes:", nb_prediction, "| Logistic Regression:", predictions[i])

# For binary classes, a positive coefficient supports classes_[1].
feature_names = tfidf.get_feature_names_out()
coefficients = lr_model.coef_[0]

positive_indices = coefficients.argsort()[-3:][::-1]
negative_indices = coefficients.argsort()[:3]

print("\n3 most positive words (support", lr_model.classes_[1], "):")
for index in positive_indices:
    print(feature_names[index], ":", round(coefficients[index], 3))

print("\n3 most negative words (support", lr_model.classes_[0], "):")
for index in negative_indices:
    print(feature_names[index], ":", round(coefficients[index], 3))

# Deployment comment:
# The dataset has only eight training examples, so test accuracy and learned words
# can change greatly with a few extra examples. I would not deploy either model yet.
# First collect more labelled reviews and evaluate on a larger, separate test set.

Logistic Regression test accuracy: 1.0

Comparison with Naive Bayes:
Fresh food and fast delivery
Naive Bayes: Positive | Logistic Regression: Positive
Late delivery and cold food
Naive Bayes: Negative | Logistic Regression: Negative
Excellent taste, will order again
Naive Bayes: Positive | Logistic Regression: Positive
Rude rider and terrible service
Naive Bayes: Negative | Logistic Regression: Negative

3 most positive words (support Positive ):
delicious : 0.217
fresh : 0.217
loved : 0.205

3 most negative words (support Negative ):
was : -0.288
late : -0.256
stale : -0.214


Logistic Regression assigns a coefficient to each TF-IDF feature. Positive coefficients support the class represented by `classes_[1]`, while negative coefficients support the class represented by `classes_[0]`.

TF-IDF is the method used to represent the text, whereas Logistic Regression performs the classification. Since the training dataset contains only eight reviews, the test accuracy and selected word coefficients may change considerably with additional data. A larger labelled dataset and a more reliable evaluation would be needed before deploying either model.